# Thai Taste Retrieval: 10k Wongnai reviews, top-10 export (task C3 input)
**Thai Food Recommendation Engine (Group: Flower)**

**Purpose.** Builds the candidate pool for the Thai taste test set, which answers professor comment
**F1** (our Thai claims were unmeasured). For each of the 24 frozen queries in `thai_queries.csv`
it takes the top 10 sentences from BM25 and the top 10 from the dense encoder and exports them to
one CSV, `topk_n10000_s42.csv`. The pooling script (C3) turns that CSV into the blind labelling sheet.

**Why 10k reviews.** In the 2k-review corpus (`thai_retrieval.ipynb`, C2), rare tastes may have
fewer than 10 relevant sentences in total, so P@10 would be capped by the corpus, not the systems.
Relevance labels are tied to the exact sentence set, so the corpus size is fixed **before**
labelling: 10,000 reviews, `SEED = 42`. Do not change these after C4 starts.

**Pipeline** (identical to the 2k notebook): PyThaiNLP `sent_tokenize(engine="crfcut")`,
whitespace normalisation, drop sentences shorter than 15 characters; BM25Okapi over `newmm` word
tokens (lower-cased, stop words kept); dense = `paraphrase-multilingual-mpnet-base-v2`, cosine on
L2-normalised embeddings.

**Colab steps**
1. *Runtime → Change runtime type → T4 GPU*.
2. If you did not clone the repo into Colab, upload `thai/thai_queries.csv` to `/content`
   (Files panel on the left).
3. *Runtime → Run all*. Wongnai is downloaded from Hugging Face; splitting takes several minutes.
4. The last cell downloads `topk_n10000_s42.csv`. Put it in `thai/results/` in the repo.

Locally the notebook also runs from the repo (reads `datasets/wongnai/train.parquet`), but the
machine is slow, so Colab is the intended runtime.


In [2]:
!git clone https://github.com/pcismyname/Food_Recommendation_Engine.git

fatal: destination path 'Food_Recommendation_Engine' already exists and is not an empty directory.


## 0. Install packages (Colab)

In [3]:
%pip install -q pythainlp python-crfsuite rank-bm25 sentence-transformers datasets pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.9/21.9 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.8 MB/s eta 0:00:00


In [4]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

IN_COLAB = Path("/content").exists()
HERE = Path.cwd()
ROOT = HERE if (HERE / "datasets").exists() else HERE.parent
DATA = ROOT / "datasets" / "wongnai"
LOCAL_REPO = (ROOT / "thai" / "thai_queries.csv").exists() and (DATA / "train.parquet").exists()

BASE  = Path("/content") if IN_COLAB and not LOCAL_REPO else ROOT / "thai"
CACHE = BASE / "cache";   CACHE.mkdir(parents=True, exist_ok=True)
OUT   = BASE / "results"; OUT.mkdir(parents=True, exist_ok=True)

N_REVIEWS  = 10000
SEED       = 42
MIN_CHARS  = 15       # drop sentence fragments shorter than this
TOP_K      = 10       # per system, per query
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
TAG = f"n{N_REVIEWS or 'all'}_s{SEED}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH  = 256 if DEVICE == "cuda" else 64   # 256 fits a T4 (16 GB) comfortably for mpnet-base
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 120)
print("colab:", IN_COLAB, "| local repo:", LOCAL_REPO, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("cache:", CACHE, "| out:", OUT, "| tag:", TAG)

colab: True | local repo: False | device: cuda Tesla T4
cache: /content/cache | out: /content/results | tag: n10000_s42


## 1. Load Wongnai and subsample
Locally: `datasets/wongnai/train.parquet`. On Colab: the same Hugging Face dataset that
`datasets/download_datasets.py` saves (`Highgroundbkk/wongnai_reviews`, train split, `to_pandas()`),
so row order matches and `review_id` = row index of the train split in both cases.

In [5]:
if (DATA / "train.parquet").exists():
    train = pd.read_parquet(DATA / "train.parquet")
    print("loaded local parquet")
else:
    from datasets import load_dataset
    train = load_dataset("Highgroundbkk/wongnai_reviews", split="train").to_pandas()
    print("loaded from Hugging Face")
assert len(train) == 40000, f"expected 40,000 train reviews, got {len(train):,}"

wg = train.reset_index(names="review_id")
if N_REVIEWS:
    wg = wg.sample(N_REVIEWS, random_state=SEED).sort_values("review_id").reset_index(drop=True)
print(wg.shape)
wg.head(3)

README.md:   0%|          | 0.00/2.16k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 26.0MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.27MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/40000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6203 [00:00<?, ? examples/s]

loaded from Hugging Face
(10000, 3)


,review_id,review_body,star_rating
0,1,อาหารที่นี่เป็นอาหารจีนแคะที่หากินยากในบ้านเรา ตัวร้านตั้งอยู่ที่ถนนพุทธมณฑลสาย 3 ไปตาม ถ.ปิ่นเกล้า-นครชัยศรี เมื่อถ...,3
1,4,อร่อย!!! เดินผ่านDigital gatewayทุกวัน ไม่ยักรู้ว่ามีร้านนี้ (เอ๊ะหรือพึ่งเปิด)\nหาง่ายมากครับ เดินจากBTSสยามเข้ามาก...,4
2,6,วันนี้ได้มีโอกาสไปนั่งซดกาแฟที่ร้านวาวี แถวๆอารีย์\n\nก็ได้ยินมาบ่อยๆละ จากหลายที่ แต่ที่บ่อยสุดก็จาก Twitter ว่ากาแ...,3


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 2. Sentence splitting
Same as the 2k notebook (`crfcut`, whitespace normalised, `MIN_CHARS` filter). Cached; this is
CPU-bound (about 1 s per 100 reviews), so expect a few minutes for 10k.

In [6]:
from pythainlp.tokenize import sent_tokenize

sent_path = CACHE / f"sentences_{TAG}.parquet"
if sent_path.exists():
    sents = pd.read_parquet(sent_path)
else:
    rows = []
    for rid, text, star in tqdm(wg[["review_id", "review_body", "star_rating"]].itertuples(index=False),
                                total=len(wg)):
        for j, s in enumerate(sent_tokenize(text, engine="crfcut")):
            s = " ".join(s.split())
            if len(s) >= MIN_CHARS:
                rows.append((rid, j, s, star))
    sents = pd.DataFrame(rows, columns=["review_id", "sent_idx", "text", "star_rating"])
    sents.to_parquet(sent_path)

print(f"{len(sents):,} sentences from {sents.review_id.nunique():,} reviews "
      f"({len(sents) / sents.review_id.nunique():.1f} per review)")

  0%|          | 0/10000 [00:00<?, ?it/s]

93,810 sentences from 10,000 reviews (9.4 per review)


## 3. Queries (C1, frozen)
Looks for `thai_queries.csv` in `/content`, the working directory, `./thai`, and the repo layout.

In [7]:
cands = [Path("/content/thai_queries.csv"), HERE / "thai_queries.csv", HERE / "thai" / "thai_queries.csv",
         ROOT / "thai" / "thai_queries.csv", Path("/content/Food_Recommendation_Engine/thai/thai_queries.csv")]
q_path = next((p for p in cands if p.exists()), None)
if q_path is None:
    raise FileNotFoundError("thai_queries.csv not found. Upload thai/thai_queries.csv from the repo to "
                            "/content (Files panel on the left), then re-run this cell.")
queries = pd.read_csv(q_path)
assert len(queries) == 24, f"expected 24 frozen queries, got {len(queries)}"
print("queries:", q_path, "|", queries.type.value_counts().to_dict())
queries

queries: /content/Food_Recommendation_Engine/thai/thai_queries.csv | {'single': 11, 'combo': 8, 'negation': 5}


,qid,query_th,query_en,taste_tags,type
0,T01,อยากกินอะไรเปรี้ยวๆ,I want something sour,sour,single
1,T02,อยากกินอะไรเผ็ดๆ แบบจัดๆ,I want something really spicy,spicy,single
2,T03,ของหวานหวานกำลังดี,desserts that are just sweet enough,sweet,single
3,T04,น้ำซุปข้นๆ เข้มข้น,thick rich broth,rich,single
4,T05,อยากกินของทอดมันๆ,I want oily fried food,oily,single
5,T06,ร้านถูกๆ แต่อร่อย,cheap but tasty place,cheap,single
6,T07,อยากกินอะไรรสจัดๆ,I want something strongly flavoured,bold,single
7,T08,อาหารรสเค็มนำ,salt-forward food,salty,single
8,T09,น้ำซุปหอมสมุนไพร,broth fragrant with herbs,herbal,single
9,T10,อาหารรสอ่อนๆ กินง่าย,mild easy-to-eat food,mild,single


## 4. BM25 on Thai word tokens
`newmm` dictionary segmentation; whitespace tokens dropped and Latin text lower-cased.
Thai stop words are **kept**, since negations like ไม่ are exactly what the taste queries depend on.

In [8]:
from pythainlp.tokenize import word_tokenize
from rank_bm25 import BM25Okapi

def th_tokenise(text):
    return [t.lower() for t in word_tokenize(text, engine="newmm", keep_whitespace=False) if t.strip()]

bm25 = BM25Okapi([th_tokenise(s) for s in tqdm(sents.text, desc="tokenise")])
print(th_tokenise(queries.query_th.iloc[10]))

tokenise:   0%|          | 0/93810 [00:00<?, ?it/s]

['ต้มยำ', 'เปรี้ยว', 'ๆ', 'เผ็ด', 'ๆ']


## 5. Dense encoder
Same model as the English baseline and the 2k notebook. Embeddings are L2-normalised and cached.

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
emb_path = CACHE / f"emb_{MODEL_NAME.split('/')[-1]}_{TAG}_min{MIN_CHARS}.npy"
if emb_path.exists():
    S = np.load(emb_path)
else:
    S = model.encode(sents.text.tolist(), batch_size=BATCH, show_progress_bar=True,
                     normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    np.save(emb_path, S)
assert len(S) == len(sents), "embedding cache does not match sentences; delete the cache files"
Q = model.encode(queries.query_th.tolist(), normalize_embeddings=True, convert_to_numpy=True)
print("sentences:", S.shape, "| queries:", Q.shape)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/367 [00:00<?, ?it/s]

sentences: (93810, 768) | queries: (24, 768)


## 6. Top-10 per system and export
One row per (query, system, rank). BM25 can score fewer than 10 sentences above zero for a query;
the top 10 by score are still exported and the `score` column shows the zero-score hits.

In [10]:
def top_bm25(qi, k=TOP_K):
    sc = bm25.get_scores(th_tokenise(queries.query_th.iloc[qi]))
    idx = np.argsort(-sc, kind="stable")[:k]
    return idx, sc[idx]

def top_dense(qi, k=TOP_K):
    sc = S @ Q[qi]
    idx = np.argsort(-sc, kind="stable")[:k]
    return idx, sc[idx]

COLS = ["qid", "query_th", "system", "rank", "sent_id", "review_id", "sent_idx", "text", "score",
        "star_rating", "n_reviews", "seed"]
rows = []
for qi, q in queries.iterrows():
    for system, fn in [("bm25", top_bm25), ("dense", top_dense)]:
        idx, sc = fn(qi)
        for r, (i, s) in enumerate(zip(idx, sc), 1):
            x = sents.iloc[i]
            rows.append((q.qid, q.query_th, system, r, f"{int(x.review_id)}_{int(x.sent_idx)}",
                         int(x.review_id), int(x.sent_idx), x.text, float(s), int(x.star_rating),
                         N_REVIEWS, SEED))
topk = pd.DataFrame(rows, columns=COLS)
assert len(topk) == len(queries) * 2 * TOP_K

out_csv = OUT / f"topk_{TAG}.csv"
topk.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("wrote", out_csv, topk.shape)
topk.head(12)

wrote /content/results/topk_n10000_s42.csv (480, 12)


,qid,query_th,system,rank,sent_id,review_id,sent_idx,text,score,star_rating,n_reviews,seed
0,T01,อยากกินอะไรเปรี้ยวๆ,bm25,1,8322_3,8322,3,และอยากกินอะไรง่ายๆ,14.698053,3,10000,42
1,T01,อยากกินอะไรเปรี้ยวๆ,bm25,2,39760_0,39760,0,อยากกินอะไรถูกๆแล้วไม่อยากออกจากบ้าน,14.002814,3,10000,42
2,T01,อยากกินอะไรเปรี้ยวๆ,bm25,3,18393_1,18393,1,อยากกินอะไรเบาๆและชิมชานมเย็น,13.848112,2,10000,42
3,T01,อยากกินอะไรเปรี้ยวๆ,bm25,4,31975_1,31975,1,ช่วงนั้นอยากทานอะไรเปรี้ยวๆหวาน เลยสั่งเป็นรสมะม่วงผสมกับเสาวรส,13.097189,3,10000,42
4,T01,อยากกินอะไรเปรี้ยวๆ,bm25,5,5457_3,5457,3,อยากกินอะไร อุ่นๆ ร้อนๆ ไปไหนดีล่ะเนี้ย...,12.867744,1,10000,42
5,T01,อยากกินอะไรเปรี้ยวๆ,bm25,6,6679_19,6679,19,ไม่เปรี้ยวอะไรมาก,12.711160,1,10000,42
6,T01,อยากกินอะไรเปรี้ยวๆ,bm25,7,13912_1,13912,1,อยากหาอะไรที่อร่อยๆทาน,12.180365,4,10000,42
7,T01,อยากกินอะไรเปรี้ยวๆ,bm25,8,17979_0,17979,0,เมียจร้า ร้องอยากกินข้าวต้มเมืองชล บอกอยากกินอะไรร้อนๆ นึกขึ้นได้ใกล้ๆ บ้านแถวทาว์นอินทาว์น,12.110510,3,10000,42
8,T01,อยากกินอะไรเปรี้ยวๆ,bm25,9,15949_0,15949,0,วันนี้หิวอยากกินอะไรง่ายๆไวๆ เลยได้มาลงที่ร้านนี้,12.017008,2,10000,42
9,T01,อยากกินอะไรเปรี้ยวๆ,bm25,10,32449_0,32449,0,อยากกินติ๋มซำ อยากกินอะไรจีนๆ เลยเข้าไปกินฮั่วเซ่งฮงซะหน่อยย ยังไม่เคยไปลองสาขาอื่นสักที,11.876060,2,10000,42


## 7. Summary

In [11]:
n_rev = sents.review_id.nunique()
print(f"sentences: {len(sents):,} from {n_rev:,} reviews ({len(sents) / n_rev:.2f} per review)")

zero = topk[topk.system == "bm25"].groupby("qid").score.apply(lambda s: int((s <= 0).sum()))
print("\nBM25 zero-score hits in top 10, per query (non-zero only):",
      zero[zero > 0].to_dict() or "none")

b = topk[topk.system == "bm25"].groupby("qid").sent_id.apply(set)
d = topk[topk.system == "dense"].groupby("qid").sent_id.apply(set)
ov = pd.DataFrame({"type": queries.set_index("qid").type,
                   "overlap@10": [len(b[q] & d[q]) for q in queries.qid],
                   "pool_size":  [len(b[q] | d[q]) for q in queries.qid]})
print(ov.groupby("type")[["overlap@10", "pool_size"]].mean().round(2))
print(f"\nExpected pool to label: {ov.pool_size.sum()} (query, sentence) pairs over {len(queries)} queries; "
      f"{topk.sent_id.nunique()} unique sentences")
ov

sentences: 93,810 from 10,000 reviews (9.38 per review)

BM25 zero-score hits in top 10, per query (non-zero only): none
          overlap@10  pool_size
type                           
combo           0.88      19.12
negation        1.20      18.80
single          1.36      18.64

Expected pool to label: 452 (query, sentence) pairs over 24 queries; 408 unique sentences


,type,overlap@10,pool_size
qid,,,
T01,single,0,20
T02,single,0,20
T03,single,1,19
T04,single,4,16
T05,single,1,19
T06,single,0,20
T07,single,1,19
T08,single,0,20
T09,single,7,13


## 8. Download the CSV (Colab only)
Then put it in `thai/results/` in the repo.

In [12]:
try:
    from google.colab import files
    files.download(str(out_csv))
except ImportError:
    print("not on Colab; CSV is at", out_csv)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>